# LegaLoom-Env: GRPO Training Notebook

**Theme: World Modeling — Professional Tasks (Theme #3.1)**

This notebook trains Qwen2.5-3B-Instruct on Indian TDS compliance using GRPO.
All curves and scores come from real training runs.

**Runtime required: T4 GPU** (Runtime → Change runtime type → T4 GPU)


In [ ]:
# Cell 1: Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install trl>=0.12.0 openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
print("Installed ✓")


In [ ]:
import subprocess, sys, os, shutil

SPACE_URL = "https://huggingface.co/spaces/aarav0202/legaloom-env"
DEST = "/content/legaloom_env"

# Force fresh clone every time
if os.path.exists(DEST):
    shutil.rmtree(DEST)

result = subprocess.run(["git", "clone", SPACE_URL, DEST], capture_output=True, text=True)
print(result.stdout or result.stderr)

sys.path.insert(0, DEST)
os.chdir(DEST)

# Verify the fix is in the new file
with open("train_grpo.py") as f:
    content = f.read()
if "(?:\\{[^{}]*\\}" in content or r'(?:\{[^{}]*\}' in content:
    print("✅ Regex fix confirmed in train_grpo.py")
else:
    print("❌ Old version — fix not present, don't proceed")

In [ ]:
# Cell 3: Load Qwen2.5-3B with Unsloth + LoRA
from unsloth import FastLanguageModel

MODEL_NAME     = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r                       = 16,
    target_modules          = ["q_proj","k_proj","v_proj","o_proj",
                               "gate_proj","up_proj","down_proj"],
    lora_alpha              = 16,
    lora_dropout            = 0,
    bias                    = "none",
    use_gradient_checkpointing = "unsloth",
    random_state            = 42,
)
print(f"Model loaded ✓  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")


In [ ]:
# Cell 4: Measure baseline BEFORE training
# Uses the same untrained model with the same prompt as training.
# rollout_episode runs a full episode and returns the terminal reward.
from unsloth import FastLanguageModel
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

FastLanguageModel.for_inference(model)

print("Measuring baseline (5 episodes per task)...")
baseline_scores = {}
for task_id in ["task_easy", "task_medium", "task_hard", "task_expert"]:
    scores = []
    for seed in range(42, 47):   # 5 seeds — fast
        env = LegaloomEnvironment()
        r = rollout_episode(model, tokenizer, env, task_id,
                            seed=seed, max_steps=8, max_new_tokens=384)
        scores.append(r["final_reward"])
        print(f"  {task_id} seed={seed}: {r['final_reward']:.3f}  steps={r['steps_used']}")
    baseline_scores[task_id] = round(sum(scores)/len(scores), 3)
    print(f"  → {task_id} avg: {baseline_scores[task_id]:.3f}\n")

print("Baseline complete ✓")
print("baseline_scores =", baseline_scores)


In [ ]:
# Cell 5: GRPO Training — full episode rollouts
# The reward function parses the model's entire action sequence and
# scores only the terminal submit_answer result. No shortcuts injected.
import sys
from train_grpo import build_training_dataset, episode_reward_fn, plot_reward_curves
from trl import GRPOConfig, GRPOTrainer

def make_reward_fn(tid):
    def fn(prompts, completions, **kwargs):
        # Strip task_id from kwargs — TRL passes dataset columns as kwargs,
        # which collides with our explicit task_id argument.
        clean = {k: v for k, v in kwargs.items() if k != "task_id"}
        return episode_reward_fn(prompts, completions, task_id=tid, **clean)
    return fn

# Switch model back to training mode
from unsloth import FastLanguageModel
FastLanguageModel.for_training(model)

all_logs  = []
result_easy = None
result_hard = None

for phase_task, phase_steps in [("task_easy", 20), ("task_hard", 20)]:
    print(f"\n{'='*55}")
    print(f"Phase: {phase_task}  ({phase_steps} steps)")
    print('='*55)

    dataset = build_training_dataset(
        task_ids=[phase_task],
        examples_per_task=max(phase_steps * 2, 40),
    )
    if dataset is None:
        print("ERROR: dataset is None — skipping")
        continue

    grpo_config = GRPOConfig(
        learning_rate              = 5e-6,
        num_train_epochs           = 1,
        per_device_train_batch_size= 2,
        gradient_accumulation_steps= 4,
        num_generations            = 4,
        max_prompt_length          = 512,
        max_completion_length      = 384,   # enough for 3–5 JSON action blocks
        beta                       = 0.01,
        logging_steps              = 1,
        max_steps                  = phase_steps,
        output_dir                 = f"./output_{phase_task}",
        report_to                  = "none",
        bf16                       = False,
        fp16                       = True,
        save_strategy              = "no",
    )

    trainer = GRPOTrainer(
        model            = model,
        args             = grpo_config,
        train_dataset    = dataset,
        reward_funcs     = [make_reward_fn(phase_task)],
        processing_class = tokenizer,
    )

    trainer.train()
    logs = trainer.state.log_history
    all_logs.extend(logs)

    rewards = [e["reward"] for e in logs if "reward" in e]
    print(f"\nPhase done. Steps with reward: {len(rewards)}")
    if rewards:
        print(f"  First reward: {rewards[0]:.3f}")
        print(f"  Last reward:  {rewards[-1]:.3f}")
        print(f"  Max reward:   {max(rewards):.3f}")

    result = {"log_history": logs, "model": model, "tokenizer": tokenizer}
    if phase_task == "task_easy":
        result_easy = result
    else:
        result_hard = result

print("\nAll training complete ✓")
print(f"Total log entries: {len(all_logs)}")

import json
with open("training_log.json", "w") as f:
    json.dump(all_logs, f, indent=2, default=str)
print("Saved → training_log.json")


In [ ]:
# Cell 6: Plot reward curves from REAL trainer.state.log_history
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result, label in [
    (axes[0], result_easy, "task_easy (Phase 1)"),
    (axes[1], result_hard,  "task_hard (Phase 2)"),
]:
    if result is None:
        ax.set_title(f"{label} — not run")
        continue
    logs    = result["log_history"]
    rewards = [e["reward"] for e in logs if "reward" in e]
    steps   = list(range(1, len(rewards) + 1))

    ax.plot(steps, rewards, "b-o", linewidth=1.5, markersize=5, alpha=0.6, label="Episode reward")

    # 3-step moving average
    if len(rewards) >= 3:
        ma = [sum(rewards[max(0,i-2):i+1]) / min(i+1, 3) for i in range(len(rewards))]
        ax.plot(steps, ma, "r-", linewidth=2.5, label="3-step moving avg")

    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Success threshold")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Episode Reward")
    ax.set_title(f"GRPO — {label}")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.suptitle("LegaLoom-Env: Real GRPO Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("reward_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reward_curves.png")


In [ ]:
# Cell 7: Measure post-training scores using rollout_episode
from unsloth import FastLanguageModel
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

FastLanguageModel.for_inference(model)

print("Measuring trained model scores (5 episodes per task)...")
trained_scores = {}
for task_id in ["task_easy", "task_medium", "task_hard", "task_expert"]:
    scores = []
    for seed in range(200, 205):   # different seeds from baseline
        env = LegaloomEnvironment()
        r = rollout_episode(model, tokenizer, env, task_id,
                            seed=seed, max_steps=8, max_new_tokens=384)
        scores.append(r["final_reward"])
        print(f"  {task_id} seed={seed}: {r['final_reward']:.3f}  steps={r['steps_used']}")
    trained_scores[task_id] = round(sum(scores)/len(scores), 3)
    print(f"  → {task_id} avg: {trained_scores[task_id]:.3f}\n")

print("Post-training eval complete ✓")
print("trained_scores =", trained_scores)


In [ ]:
# Cell 8: Before / After comparison bar chart
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json

tasks  = ["task_easy", "task_medium", "task_hard", "task_expert"]
labels = ["Easy", "Medium", "Hard", "Expert"]
before = [baseline_scores[t] for t in tasks]
after  = [trained_scores[t]  for t in tasks]

x   = range(len(tasks))
fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar([i - 0.2 for i in x], before, 0.35,
            label="Before GRPO (untrained)", color="#e74c3c", alpha=0.85)
b2 = ax.bar([i + 0.2 for i in x], after,  0.35,
            label="After GRPO (20+20 steps)", color="#2ecc71", alpha=0.85)

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Average Score (5 episodes)", fontsize=11)
ax.set_title("LegaLoom-Env: Before vs After GRPO Training", fontsize=13, fontweight="bold")
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Success threshold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → before_after.png")

# Save scores
with open("training_scores.json", "w") as f:
    json.dump({"baseline": baseline_scores, "trained": trained_scores}, f, indent=2)
print("Saved → training_scores.json")

print()
print("="*40)
print(f"Baseline avg : {sum(before)/4:.3f}")
print(f"Trained avg  : {sum(after)/4:.3f}")
print(f"Lift         : {(sum(after)-sum(before))/4:+.3f}")
print("="*40)


In [ ]:
# Cell 9: Download output files back to your machine
from google.colab import files

for fname in ["reward_curves.png", "before_after.png", "training_scores.json", "training_log.json"]:
    try:
        files.download(fname)
        print(f"Downloaded → {fname}")
    except Exception as e:
        print(f"Could not download {fname}: {e}")

print()
print("Next steps:")
print("1. Add these files to your git repo")
print("2. Update README.md table with the real scores above")
print("3. Update blog_post.md Results section with real numbers")
print("4. git add reward_curves.png before_after.png training_scores.json && git commit && git push")


In [ ]:
# Cell 10: Save adapter weights (optional — push to HF Hub)
result_hard["model"].save_pretrained("legaloom_qwen25_3b_grpo")
result_hard["tokenizer"].save_pretrained("legaloom_qwen25_3b_grpo")
print("Saved adapter → legaloom_qwen25_3b_grpo/")

# Uncomment to push trained model to HF:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# result_hard["model"].push_to_hub("aarav0202/legaloom-qwen25-3b-grpo")
# result_hard["tokenizer"].push_to_hub("aarav0202/legaloom-qwen25-3b-grpo")
